In [1]:
import gradio as gr
from PIL import Image
import os
from pathlib import Path
import base64
from openai import OpenAI
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Thread-safe counter for progress tracking
class ProgressTracker:
    def __init__(self, total):
        self.total = total
        self.completed = 0
        self.successful = 0
        self.failed = 0
        self.lock = threading.Lock()
    
    def increment(self, success=True):
        with self.lock:
            self.completed += 1
            if success:
                self.successful += 1
            else:
                self.failed += 1
    
    def get_progress(self):
        with self.lock:
            return self.completed, self.successful, self.failed

def enhance_image_with_openai(client, image_path, output_path, prompt, quality, size):
    """
    Verbessert ein Bild mit OpenAI GPT Image
    """
    with open(image_path, "rb") as img_file:
        result = client.images.edit(
            model="gpt-image-1",
            image=[img_file],
            prompt=prompt,
            quality=quality,
            size=size
        )
    
    image_base64 = result.data[0].b64_json
    image_bytes = base64.b64decode(image_base64)
    
    with open(output_path, "wb") as f:
        f.write(image_bytes)
    
    return True

def process_single_image(args):
    """
    Verarbeitet ein einzelnes Bild - für parallele Ausführung
    """
    idx, image_path, client, output_folder, enhancement_prompt, quality, size, token_costs = args
    
    result = {
        'idx': idx,
        'name': image_path.name,
        'success': False,
        'messages': [],
        'cost': 0
    }
    
    try:
        result['messages'].append(f"🔄 Verarbeite: {image_path.name}")
        
        # Original-Bildgröße
        with Image.open(image_path) as img:
            original_size = img.size
            result['messages'].append(f"   📐 Original: {original_size[0]}x{original_size[1]}")
        
        # Ausgabe-Pfad
        output_path = Path(output_folder) / f"enhanced_{image_path.stem}.png"
        
        # Bild mit OpenAI verbessern
        enhance_image_with_openai(
            client, 
            image_path, 
            output_path, 
            enhancement_prompt,
            quality,
            size
        )
        
        # Neue Größe
        with Image.open(output_path) as img:
            new_size = img.size
            result['messages'].append(f"   ✅ Verbessert: {new_size[0]}x{new_size[1]}")
            result['messages'].append(f"   💾 Gespeichert: {output_path.name}")
        
        # Kosten schätzen
        est_tokens = token_costs.get(quality, {}).get(size, 1000)
        est_cost = (est_tokens / 1000) * 0.04
        result['cost'] = est_cost
        result['success'] = True
        
    except Exception as e:
        error_msg = str(e)
        result['messages'].append(f"   ❌ Fehler: {error_msg}")
        
        if "rate_limit" in error_msg.lower() or "429" in error_msg:
            result['messages'].append(f"   ⚠️ Rate Limit erreicht! Reduziere parallele Worker.")
        elif "insufficient_quota" in error_msg.lower():
            result['messages'].append(f"   ⚠️ API-Quota aufgebraucht! Prüfe dein OpenAI-Guthaben.")
    
    return result

def process_images(folder_path, output_folder, api_key, enhancement_prompt, quality, size, parallel_workers, delay_between_batches):
    """
    Verarbeitet alle Bilder in einem Ordner mit OpenAI GPT Image (parallel)
    """
    if not api_key:
        return "❌ Bitte gib deinen OpenAI API-Schlüssel ein!\n\n📝 So geht's:\n1. Gehe zu https://platform.openai.com/api-keys\n2. Erstelle einen API-Schlüssel\n3. Füge ihn hier ein"
    
    if not os.path.exists(folder_path):
        return "❌ Der angegebene Ordner existiert nicht!"
    
    # OpenAI Client konfigurieren
    try:
        client = OpenAI(api_key=api_key)
    except Exception as e:
        return f"❌ Fehler bei der API-Konfiguration: {str(e)}"
    
    # Ausgabe-Ordner erstellen
    if not output_folder:
        output_folder = os.path.join(folder_path, "enhanced_openai")
    os.makedirs(output_folder, exist_ok=True)
    
    # Unterstützte Bildformate
    image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    
    # Alle Bilder im Ordner finden
    image_files = [f for f in Path(folder_path).iterdir() 
                   if f.suffix.lower() in image_extensions and f.is_file()]
    
    if not image_files:
        return "❌ Keine Bilder im angegebenen Ordner gefunden!"
    
    results_output = [f"🚀 Starte KI-Verbesserung von {len(image_files)} Bildern mit OpenAI GPT Image...\n"]
    results_output.append(f"⚙️ Qualität: {quality} | Größe: {size}")
    results_output.append(f"🔄 Parallele Worker: {parallel_workers} | Batch-Verzögerung: {delay_between_batches}s\n")
    
    # Token-Kosten basierend auf Qualität und Größe
    token_costs = {
        "low": {"1024x1024": 272, "1024x1536": 408, "1536x1024": 400, "auto": 350},
        "medium": {"1024x1024": 1056, "1024x1536": 1584, "1536x1024": 1568, "auto": 1400},
        "high": {"1024x1024": 4160, "1024x1536": 6240, "1536x1024": 6208, "auto": 5200},
        "auto": {"1024x1024": 4160, "1024x1536": 6240, "1536x1024": 6208, "auto": 5200}
    }
    
    # Progress Tracker
    tracker = ProgressTracker(len(image_files))
    total_cost = 0
    
    # Vorbereitung der Aufgaben
    tasks = [
        (idx, image_path, client, output_folder, enhancement_prompt, quality, size, token_costs)
        for idx, image_path in enumerate(image_files, 1)
    ]
    
    # Parallele Verarbeitung
    results_output.append(f"⏳ Verarbeite {len(image_files)} Bilder mit {parallel_workers} parallelen Workern...\n")
    
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=parallel_workers) as executor:
        # Submit all tasks
        futures = {executor.submit(process_single_image, task): task for task in tasks}
        
        # Process completed tasks
        for future in as_completed(futures):
            result = future.result()
            
            # Update tracker
            tracker.increment(success=result['success'])
            total_cost += result['cost']
            
            # Add messages to output
            results_output.append(f"\n[{result['idx']}/{len(image_files)}] {result['name']}")
            results_output.extend(result['messages'])
            
            # Show progress every few images
            completed, successful, failed = tracker.get_progress()
            if completed % max(1, parallel_workers) == 0 or completed == len(image_files):
                progress_pct = int(completed / len(image_files) * 100)
                results_output.append(f"\n📈 Fortschritt: {completed}/{len(image_files)} ({progress_pct}%) | ✅ {successful} | ❌ {failed}")
            
            # Batch delay - pause after each batch of workers completes
            if completed % parallel_workers == 0 and completed < len(image_files) and delay_between_batches > 0:
                results_output.append(f"⏳ Batch abgeschlossen. Warte {delay_between_batches}s...")
                time.sleep(delay_between_batches)
    
    elapsed_time = time.time() - start_time
    
    # Final summary
    completed, successful, failed = tracker.get_progress()
    
    summary = f"\n\n{'='*60}\n"
    summary += "📊 ZUSAMMENFASSUNG\n"
    summary += f"{'='*60}\n"
    summary += f"✅ Erfolgreich: {successful}\n"
    summary += f"❌ Fehlgeschlagen: {failed}\n"
    summary += f"⏱️ Gesamtzeit: {elapsed_time:.1f}s ({elapsed_time/len(image_files):.1f}s pro Bild)\n"
    summary += f"⚡ Speedup: ~{len(image_files)/(elapsed_time/parallel_workers):.1f}x durch {parallel_workers} Worker\n"
    summary += f"💾 Gespeichert in: {output_folder}\n"
    summary += f"💰 Geschätzte Kosten: ${total_cost:.3f}\n"
    summary += f"{'='*60}\n\n"
    summary += "💡 TIPP: Bei Rate Limits die Worker-Anzahl reduzieren oder Batch-Verzögerung erhöhen!"
    
    return "\n".join(results_output) + summary

def create_interface():
    with gr.Blocks(title="OpenAI GPT Image Enhancer (Parallel)", theme=gr.themes.Soft()) as app:
        gr.Markdown("# 🤖 OpenAI GPT-5 Image Enhancer ⚡ PARALLEL")
        gr.Markdown("**Powered by gpt-image-1** - Jetzt mit Multi-Threading für bis zu 5x schnellere Verarbeitung!")
        
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🔑 API-Konfiguration")
                
                api_key_input = gr.Textbox(
                    label="OpenAI API-Schlüssel",
                    type="password",
                    placeholder="sk-...",
                    info="Hole ihn von: https://platform.openai.com/api-keys"
                )
                
                gr.Markdown("### 📁 Ordner-Einstellungen")
                
                folder_input = gr.Textbox(
                    label="Bildordner-Pfad",
                    placeholder="C:/Users/Name/Bilder",
                    info="Ordner mit den zu verbessernden Bildern"
                )
                
                output_folder_input = gr.Textbox(
                    label="Ausgabe-Ordner (optional)",
                    placeholder="Leer lassen für 'enhanced_openai'",
                    info="Wo sollen die verbesserten Bilder gespeichert werden?"
                )
                
                gr.Markdown("### ⚙️ Qualitäts-Einstellungen")
                
                quality_dropdown = gr.Dropdown(
                    choices=["low", "medium", "high", "auto"],
                    value="high",
                    label="🎨 Bildqualität",
                    info="high = beste Qualität (teurer) | low = schneller (günstiger)"
                )
                
                size_dropdown = gr.Dropdown(
                    choices=["1024x1024", "1024x1536", "1536x1024", "auto"],
                    value="auto",
                    label="📐 Ausgabe-Größe",
                    info="auto = KI wählt optimale Größe"
                )
                
                gr.Markdown("### ⚡ Performance-Einstellungen (NEU!)")
                
                workers_slider = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=5,
                    step=1,
                    label="🔄 Parallele Worker",
                    info="Anzahl gleichzeitig verarbeiteter Bilder (5 empfohlen)"
                )
                
                batch_delay_slider = gr.Slider(
                    minimum=0,
                    maximum=5,
                    value=1,
                    step=0.5,
                    label="⏱️ Verzögerung zwischen Batches (Sekunden)",
                    info="Pause nach jedem Batch. Verhindert Rate Limits."
                )
                
                gr.Markdown("### ✨ Verbesserungs-Prompt")
                
                prompt_input = gr.Textbox(
                    label="AI-Prompt für Bildverbesserung",
                    value="""Enhance this image to professional quality with the following improvements:
- Dramatically increase sharpness and resolution
- Improve colors, contrast and vibrancy
- Remove all noise, compression artifacts and imperfections
- Add realistic details and fine textures
- Perfect the lighting and exposure
- Make the image look crisp, clear and professional
- The object, drawings or anything in the image should be exactly the same. 
- The background of image shoudl always be white. 
Keep the exact same content and composition.""",
                    lines=8,
                    info="Beschreibe, wie OpenAI die Bilder verbessern soll"
                )
                
                process_btn = gr.Button(
                    "🚀 BILDER JETZT VERBESSERN (PARALLEL)",
                    variant="primary",
                    size="lg"
                )
            
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Status & Ergebnisse")
                
                output_text = gr.Textbox(
                    label="Verarbeitungsstatus",
                    lines=30,
                    max_lines=40,
                    show_label=False,
                    placeholder="Ergebnisse erscheinen hier...",
                    autoscroll=True
                )
        
        with gr.Row():
            with gr.Column():
                gr.Markdown("""
                ### ⚡ Schnellstart (Parallel Mode)
                
                1. **API-Schlüssel holen** (2 Minuten):
                   - Gehe zu [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
                   - Klicke auf "Create new secret key"
                   - Kopiere den Schlüssel (beginnt mit `sk-`)
                   
                2. **Schlüssel eingeben**: Füge ihn oben ein
                
                3. **Ordner wählen**: Gib deinen Bildordner-Pfad ein
                
                4. **Performance einstellen** (NEU!):
                   - **5 Worker** = 5x schneller! 🚀
                   - **1s Batch-Delay** = verhindert Rate Limits
                   - Bei Problemen: Worker reduzieren
                   
                5. **START**: Klicke auf "BILDER JETZT VERBESSERN"
                
                ### 💰 Kosten (unverändert)
                
                - **Low Quality**: ~$0.01-0.02 pro Bild
                - **Medium Quality**: ~$0.04-0.06 pro Bild  
                - **High Quality**: ~$0.15-0.25 pro Bild
                - **Neue Accounts**: Oft $5 gratis Credits! 🎉
                
                **Beispiel**: 20 Bilder (high, 5 Worker) = ~$4.00 in ~2-3 Minuten
                """)
            
            with gr.Column():
                gr.Markdown("""
                ### ✨ Features (ERWEITERT)
                
                - 🤖 **GPT-Image-1**: Neueste OpenAI Bild-KI
                - ⚡ **Multi-Threading**: Bis zu 10 Bilder parallel
                - 🚀 **5x schneller**: 5 Worker = 5x Speed!
                - 🎨 **Intelligente Verbesserung**: World-knowledge
                - 📐 **Flexible Größen**: Square, Portrait, Landscape
                - 💎 **Premium Qualität**: High-fidelity Upscaling
                - 🔄 **Batch-Verarbeitung**: Alle Bilder automatisch
                - 📊 **Live-Progress**: Echtzeit Fortschrittsanzeige
                
                ### ⚙️ Performance-Tipps
                
                - **Start mit 5 Workern**: Optimale Balance
                - **Bei Rate Limits**: Worker auf 2-3 reduzieren
                - **Große Batches**: 1-2s Batch-Delay setzen
                - **Schnellster Mode**: 10 Worker + 0s Delay (riskant!)
                - **Sicherer Mode**: 3 Worker + 2s Delay
                
                ### ⚠️ Rate Limits
                
                - OpenAI erlaubt ~50-100 Requests/min
                - Mit 5 Workern: ~300 Bilder/Stunde möglich
                - Bei Limits: Automatische Fehlerbehandlung
                """)
        
        with gr.Row():
            gr.Markdown("""
            ---
            ### 🔥 Performance-Vergleich
            
            | Bilder | Sequential | 3 Worker | 5 Worker | 10 Worker |
            |--------|------------|----------|----------|-----------|
            | 10 Bilder | ~10 min | ~4 min | ~2.5 min | ~1.5 min |
            | 50 Bilder | ~50 min | ~18 min | ~11 min | ~6 min |
            | 100 Bilder | ~100 min | ~35 min | ~22 min | ~12 min |
            
            **Mit 5 Workern bis zu 5x schneller!** ⚡ Rate Limits beachten!
            """)
        
        process_btn.click(
            fn=process_images,
            inputs=[folder_input, output_folder_input, api_key_input, prompt_input, 
                   quality_dropdown, size_dropdown, workers_slider, batch_delay_slider],
            outputs=output_text
        )
    
    return app

if __name__ == "__main__":
    app = create_interface()
    print("🚀 Starte OpenAI GPT-5 Image Enhancer (PARALLEL)...")
    print("📦 Installation: pip install gradio openai pillow")
    print("🔑 API-Schlüssel: https://platform.openai.com/api-keys")
    print("⚡ NEU: Multi-Threading Support für bis zu 10x schnellere Verarbeitung!")
    app.launch()

c:\Users\r.musawi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Starte OpenAI GPT-5 Image Enhancer (PARALLEL)...
📦 Installation: pip install gradio openai pillow
🔑 API-Schlüssel: https://platform.openai.com/api-keys
⚡ NEU: Multi-Threading Support für bis zu 10x schnellere Verarbeitung!
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
